This notebook is used to "unit" test the ellipsoidal and reflection effects against simple PHOEBE systems to ensure behavior is correct. This is done by generating the lightcurves for various combinations of orbital parameters and then verifing that eBEER can replicate this result either implicitly or with fitting for reflection. 

This notebook is useful for quick tests of few combinations of orbital parameters to verify behaviors or other quick tests. For longer tests use "PHOEBEUnitTestingScript.py".

It is of note that the testing for reflection effect also includes the ellipsoidal effect so it can only be used as a confirmation of reflection assuming the ellipsoidal effect works. I do this as my code in "phoebe_to_eBEER.py" does not readily have a way to turn of the ellipsoidal effect because the $\alpha$ scaling coefficients are calculated from the linear limb and gravity darkening coefficients. It would not be too difficult to add a new boolean to toggle ellipsoidal but, I wanted to avoid messing with my code in "phoebe_to_eBEER.py" unless necessary.

We start by importing the necessary packages:
 - numpy, pandas, matplotlib
 - phoebe
 - astropy.units - used to ensure PHOEBE understand what units we are working with
 - scipy.optimize.curve_fit - for fitting
 - testing_utils - a .py file with some general tools for constructing and printing PHOEBE binaries
 - phoebe_to_eBEER - .py file containing model code to take a phoebe binary and return eBEER model light curve

In [ ]:
import phoebe

import sys
sys.path.insert(1, '../scripts')
import phoebe_to_eBEER
import testing_utils as utils

import numpy as np
import matplotlib.pyplot as plt
from astropy import units as u
from scipy.optimize import curve_fit

import itertools
import time

We start by making a list of various parameters we would like to test for. If you would like to look at specific system that are not combinations of the same sets then you must set comb_list by hand.

We can easily toggle the reflection effect (and necessary testing steps like fitting) with the `Refl_ON` boolean

In [ ]:
eccs = (0.8,)#(0, 0.3, 0.8,)
incls = (0, 30, 60, 90, 120, 150, 180,)
per0s = (30,)#(0, 30, 90, 150, 180, 240,)
qs = (1,)#(1, 0.5, 2,)
ps = (1,)#(1, 5, 10, 15, 20,)

combinations = itertools.product(eccs, incls, per0s, ps, qs)
comb_list = list(combinations)
# comb_list = ((0,0,0,1,1),)

Refl_ON = False

PHOEBE returns absolute flux so we must convert this to relative flux with the flux of the system with all effects turned off as the reference (this is the modulating eBEER calculates). To do this we initialize the first phoebe_binary and simulate the lightcurve with the effects on and then initialize the second binary and simulate its light curve with all effects off.

We set passband to Kepler and use Phoenix as the atmosphere model. We can also conveniently turn of eclipses with `'eclipse_method'='only_horizon'`.
We can turn off reflection with `'irrad_method'='none'` and ellipsoidal with `'distortion_method'='sphere'`. We have to turn off ellipsoidal for both stars which can be easily accomploshed with `set_value_all()`.
The last high level modification of note is `'ntriangles'`. For systems with $Porb < 1$ 10k triangles is more than sufficient. For higher orbital periods more triangles (100k) is needed to not lose the signal to noise. However even this stops working at $Porb >= 100$. Make sure to use the same ntriangles for both phoebe binaries.

A few notes on certain choices made in the following algorithm:
- I keep the following all in one cell so that the PHOEBE binaries are recreated each time we test different combinations to avoid any issues where the object is already in memory.
- I chose to set the values in the loop each time as opposed to using the get_phoebe_binary function I have already created as this is significantly faster. phoebe.default_binary() takes a few seconds so creating a new binary for each combination would be inefficient.
- I set the stellar radii to their max according to PHOEBE as sometimes- for higher eccentricities- q**0.8 is even larger and would have the star overflowing at periastron

In [ ]:
start = time.time()

# Initialize Binary with Effects ON
b_ = phoebe.default_binary()
b_.add_dataset('lc', times=0, label= 'lc01')
b_.set_value_all('gravb_bol', 0.32)
b_.set_value_all('ld_mode*', 'manual')
b_.set_value_all('ld_func*', 'linear')
b_.set_value_all('ld_coeffs*', 0.5)
b_.set_value_all('ntriangles', 10000)
b_['eclipse_method'] = 'only_horizon'
b_['passband'] = 'Kepler:mean'
b_.set_value_all('atm', 'phoenix')
b_.flip_constraint('mass@primary', solve_for='sma')
if not Refl_ON:
    b_.set_value('irrad_method', 'none')

# Initialize Binary with Effects OFF
b_2 = phoebe.default_binary()
b_2.add_dataset('lc', times=0, label= 'lc01')
b_2.set_value_all('gravb_bol', 0.32)
b_2.set_value_all('ld_mode*', 'manual')
b_2.set_value_all('ld_func*', 'linear')
b_2.set_value_all('ld_coeffs*', 0.5)
b_2.set_value_all('ntriangles', 10000)
b_2['eclipse_method'] = 'only_horizon'
b_2['passband'] = 'Kepler:mean'
b_2.set_value_all('atm', 'phoenix')
b_2.flip_constraint('mass@primary', solve_for='sma')
b_2.set_value('irrad_method', 'none')
b_2.set_value_all('distortion_method', value='sphere')


def fit_scaling(arefl1, arefl2):
    scaled = np.zeros(2)
    scaled[0] = arefl1 * 7
    scaled[1] = arefl2 * 7
    return scaled

def fit_eBEER(t, arefl1, arefl2):
    scaled = fit_scaling(arefl1, arefl2)
    flux = phoebe_to_eBEER.get_eBEER_lc(b_, t, 0, 0, scaled[0], scaled[1])
    return flux


periods = np.zeros(len(comb_list))
PHOEBE_Ampl = np.zeros(len(comb_list))
eBEER_Ampl = np.zeros(len(comb_list))
RMSDs = np.zeros(len(comb_list))

for i, comb in enumerate(comb_list):
    start_sys = time.time()
    ecc, incl, per0, p, q = comb
    comb_str = f'lc_{p}_{int(ecc*10)}_{incl}_{per0}_{int(q*10)}'
    print('System:', comb_str)

    # Configure Binary with Effects ON
    times = np.linspace(0,p,101)
    b_['lc01@dataset@times']= times
    b_['orbit@period'].set_value(p * u.day)
    b_['orbit@incl'].set_value(incl * u.deg)
    b_['orbit@ecc'].set_value(ecc)
    b_['orbit@per0'].set_value(per0 * u.deg)
    b_['mass@primary@component'].set_value(1 * u.M_sun)
    b_['orbit@q'].set_value(q)
    b_.run_checks_compute()
    R1_max = b_['primary@component@requiv_max'].get_value(u.R_sun)
    R2_max = b_['secondary@component@requiv_max'].get_value(u.R_sun)
    b_['primary']['requiv'].set_value(R1_max * u.R_sun)
    b_['secondary']['requiv'].set_value(R2_max * u.R_sun)
    # b_['primary']['requiv'].set_value(1 * u.R_sun)
    # b_['secondary']['requiv'].set_value(q**0.8 * u.R_sun)
    setup_done = time.time()
    print('\tSystem Setup Duration:', setup_done - start_sys)
    utils.print_pb_params(b_)

    # try: # Will hide any combinations that are failing due to falling outside 
         # of valid atmosphere tables or similar issues
    b_.run_compute(model= comb_str, overwrite= True)
    lc_done = time.time()
    print('\tLC Dataset Duration:', lc_done - setup_done)
    # except:
    #     print('\tFAILED COMPUTE')
    #     continue

    # Configure Binary with Effects OFF
    b_2['orbit@period'].set_value(p * u.day)
    b_2['orbit@incl'].set_value(incl * u.deg)
    b_2['orbit@ecc'].set_value(ecc)
    b_2['orbit@per0'].set_value(per0 * u.deg)
    b_2['mass@primary@component'].set_value(1 * u.M_sun)
    b_2['orbit@q'].set_value(q)
    b_2['primary']['requiv'].set_value(R1_max * u.R_sun)
    b_2['secondary']['requiv'].set_value(R2_max * u.R_sun)
    # b_2['primary']['requiv'].set_value(1 * u.R_sun)
    # b_2['secondary']['requiv'].set_value(q**0.8 * u.R_sun)
    # utils.print_pb_params(b_2)
    b_2.run_compute(overwrite= True)

    flux = b_[f'fluxes@lc01@{comb_str}@model'].value
    ref_flux = b_2['fluxes@lc01@latest@model'].value
    relative_flux_ = (flux - ref_flux)/ref_flux * 1e6


    if not Refl_ON:
        eBEER_flux = phoebe_to_eBEER.get_eBEER_lc(b_, times, 0, 0, 0, 0)
    else:
        popt_, pcov = curve_fit(fit_eBEER, times, relative_flux_, p0= (0.5, 0.5), bounds= ([0, 0], [1, 1]))
        rescaled = fit_scaling(popt_[0], popt_[1])
        fit_flux_ = fit_eBEER(times, popt_[0], popt_[1])
        print('\tFit Duration:', time.time() - lc_done)
        print('\t\t', rescaled)

    # Calculate and store RMSDs and Amplitudes
    if not Refl_ON: residuals = (relative_flux_ - eBEER_flux)
    else: residuals = (relative_flux_ - fit_flux_)
    
    MSD = np.mean(residuals**2)
    RMSD = np.sqrt(MSD)
    RMSDs[i] = RMSD
    periods[i] = p

    if not Refl_ON:
        eBEER_Ampl[i] = (np.max(eBEER_flux) - np.min(eBEER_flux)) / 2
    else:
        eBEER_Ampl[i] = (np.max(fit_flux_) - np.min(fit_flux_)) / 2
    PHOEBE_Ampl[i] = (np.max(relative_flux_) - np.min(relative_flux_)) / 2
    

    print('\tTotal Duration:', time.time() - start_sys)
    
    # Plot Combination
    t0 = b_['orbit@component@t0_perpass'].get_value(u.day)
    phase = (times - t0) % p
    plt.figure(figsize=(8, 5))
    plt.plot(times, relative_flux_, 'b.', markersize=1, label= 'PHOEBE LC')
    if not Refl_ON:
        plt.plot(times, eBEER_flux, 'g', markersize=1, label= 'eBEER LC')
    else:
        plt.plot(phase, fit_flux_, 'r', markersize=1, label= 'eBEER Fit')
    plt.title(fr'{i} - ecc:{ecc}, incl:{incl}, $\omega$:{per0}, Porb:{p}, q:{q}')
    plt.xlabel('Time (days)')
    plt.ylabel('dFlux (ppm)')
    plt.legend()
    plt.show()

In the above cell we calculate the amplitudes of each light curve and the Root Mean-Squared Difference (RMSD) and plot them below

In [ ]:
plt.scatter(periods, RMSDs)
plt.xscale('log')
plt.xlabel('Period (days)')
plt.ylabel('RMSD (ppm)')
plt.title('sys_config')
# plt.legend()

In [ ]:
plt.plot(periods, eBEER_Ampl, 'b.', label= 'eBEER')
plt.plot(periods, PHOEBE_Ampl, 'r.', label= 'PHOEBE')
plt.xscale('log')
plt.yscale('log')
plt.xlabel('Period (days)')
plt.ylabel('Amplitude (ppm)')
plt.title('LC Amplitude vs Orbital Period')
plt.legend()